# Interpretacja modelu — Grad-CAM + Saliency Maps (v2 — naprawiony)

### Co było nie tak w v1?
Backbone (EfficientNetB0) jako sub-model tworzy **oddzielny graf obliczeniowy**.
Próba obliczenia gradientów przez granicę dwóch grafów dawała `None`.

**Naprawa:** Zamiast szukać wewnętrznych warstw conv w backbone, używamy `backbone.output` — tensor (7×7×1280)
który **jest** w grafie głównego modelu. Budujemy jeden `grad_model` z dwoma wyjściami:
`[feature_maps, prediction]` w jednym przejściu.

### Metody

| Metoda | Co pokazuje | Rozdzielczość |
|--------|-------------|---------------|
| **Grad-CAM** | Które *regiony* obrazu zdecydowały | Niska (7×7 → upscaled) |
| **Grad-CAM++** | Lepsza lokalizacja przy wielu artefaktach | Niska |
| **Vanilla Saliency** | Które *piksele* mają największy wpływ | Per-piksel (224×224) |
| **SmoothGrad** | Wygładzona saliency — czytelniejsza | Per-piksel |


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import os, io

tf.get_logger().setLevel('ERROR')
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
IMG_SIZE = 224

MODEL_PATH = "saved_models/best_model.keras"
# MODEL_PATH = "saved_models/final_model.keras"

LABEL_MAP = {0: "Real", 1: "AI-Generated"}

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Brak modelu: {MODEL_PATH}\n"
        f"Pliki: {os.listdir('saved_models') if os.path.isdir('saved_models') else 'BRAK FOLDERU'}"
    )

model = tf.keras.models.load_model(MODEL_PATH)
print(f"✅ Model wczytany: {MODEL_PATH}")
print(f"   Parametry: {model.count_params():,}")
print(f"   Layers: {[l.name for l in model.layers]}")

## 1. Budowa gradient model

**Problem Keras 3:** `tf.keras.Model(model.input, [backbone.output, model.output])` nie działa,
bo backbone tworzy oddzielny podgraf. `KeyError` przy próbie trace'owania.

**Rozwiązanie:** Budujemy NOWY graf od zera, ale z tymi samymi warstwami (współdzielone wagi).
Nowe `Input` → `backbone(input)` → head layers → dwa wyjścia w jednym grafie.


In [ ]:
# ── Znajdź backbone i zbuduj grad_model ──

backbone = None
backbone_name = None
for layer in model.layers:
    if hasattr(layer, 'layers') and len(layer.layers) > 10:
        backbone = layer
        backbone_name = layer.name
        break

if backbone is None:
    raise RuntimeError("Nie znaleziono sub-modelu backbone!")

print(f"Backbone: {backbone.name} ({len(backbone.layers)} warstw)")

# ── Zbierz warstwy head (po backbone) ──
head_layers = []
found_backbone = False
for layer in model.layers:
    if layer.name == backbone_name:
        found_backbone = True
        continue
    if found_backbone and not isinstance(layer, tf.keras.layers.InputLayer):
        head_layers.append(layer)

print(f"Head layers: {[l.name for l in head_layers]}")

# ── Budowa NOWEGO grafu z TYMI SAMYMI warstwami ──
# Nowy Input → backbone() → feature_maps
#                         → head layers → prediction
# Współdzielone warstwy = współdzielone wagi
# Jeden graf = gradienty działają

inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='grad_input')
feature_maps = backbone(inp)  # (batch, 7, 7, 1280)

x = feature_maps
for layer in head_layers:
    x = layer(x)
prediction = x  # (batch, 1)

grad_model = tf.keras.Model(inputs=inp, outputs=[feature_maps, prediction])

# ── Test ──
test_input = np.random.uniform(0, 255, (1, IMG_SIZE, IMG_SIZE, 3)).astype(np.float32)
test_feat, test_pred = grad_model(test_input, training=False)
print(f"\n✅ Gradient model OK!")
print(f"   Feature maps: {test_feat.shape}")
print(f"   Prediction:   {test_pred.shape} → {test_pred.numpy()[0][0]:.4f}")

# ── Test gradientów ──
with tf.GradientTape() as tape:
    feat, pred = grad_model(test_input, training=False)
    loss = pred[0, 0]
grads = tape.gradient(loss, feat)
if grads is not None:
    print(f"✅ Gradienty OK! shape={grads.shape}, mean={tf.reduce_mean(tf.abs(grads)).numpy():.6f}")
else:
    print("❌ Gradienty = None — coś jest nie tak")

In [ ]:
def preprocess(image_pil):
    img = image_pil.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return np.array(img, dtype=np.float32)  # [0, 255]

def predict(model, img_array):
    return model.predict(img_array[np.newaxis], verbose=0)[0][0]

In [ ]:
# ═══════════════════════════════════════════
# Grad-CAM — używa grad_model (jeden graf!)
# ═══════════════════════════════════════════

def compute_gradcam(grad_model, img_array):
    """
    Grad-CAM na feature maps backbone'a.

    grad_model zwraca [feature_maps (7,7,1280), prediction (1,)].
    Oba wyjścia są w jednym grafie → gradienty działają.
    """
    img_tensor = tf.cast(img_array[np.newaxis], tf.float32)

    with tf.GradientTape() as tape:
        feature_maps, prediction = grad_model(img_tensor, training=False)
        # tape automatycznie śledzi feature_maps bo jest outputem modelu
        loss = prediction[0, 0]

    # Gradienty predykcji względem feature maps
    grads = tape.gradient(loss, feature_maps)

    if grads is None:
        print("⚠️ Gradienty None — coś poszło nie tak z grafem")
        return np.zeros((IMG_SIZE, IMG_SIZE))

    # Wagi: średni gradient per feature map
    weights = tf.reduce_mean(grads, axis=(1, 2))  # (1, 1280)

    # Ważona suma feature maps
    cam = tf.reduce_sum(feature_maps[0] * weights[0], axis=-1)  # (7, 7)

    # ReLU + normalizacja
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)

    # Upscale do 224×224
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_SIZE, IMG_SIZE))[..., 0]

    return cam.numpy()


def compute_gradcam_pp(grad_model, img_array):
    """
    Grad-CAM++: wagi 2. rzędu → lepsza lokalizacja rozproszonych artefaktów.
    """
    img_tensor = tf.cast(img_array[np.newaxis], tf.float32)

    with tf.GradientTape(persistent=True) as tape:
        feature_maps, prediction = grad_model(img_tensor, training=False)
        tape.watch(feature_maps)  # jawne watch dla persistent tape
        loss = prediction[0, 0]

    grads_1 = tape.gradient(loss, feature_maps)

    if grads_1 is None:
        print("⚠️ Gradienty None")
        del tape
        return np.zeros((IMG_SIZE, IMG_SIZE))

    # Grad-CAM++ wagi: α = relu(grads)^2 / (2*grads^2 + sum(fmap * grads^3) + eps)
    grads_2 = grads_1 ** 2
    grads_3 = grads_1 ** 3
    sum_fmap_grads3 = tf.reduce_sum(feature_maps * grads_3, axis=(1, 2), keepdims=True)

    denom = 2.0 * grads_2 + sum_fmap_grads3 + 1e-8
    alpha = grads_2 / denom

    weights = tf.reduce_sum(alpha * tf.nn.relu(grads_1), axis=(1, 2))  # (1, 1280)

    cam = tf.reduce_sum(feature_maps[0] * weights[0], axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_SIZE, IMG_SIZE))[..., 0]

    del tape
    return cam.numpy()


# ── Szybki test ──
test_img = np.random.uniform(0, 255, (IMG_SIZE, IMG_SIZE, 3)).astype(np.float32)
cam_test = compute_gradcam(grad_model, test_img)
cam_pp_test = compute_gradcam_pp(grad_model, test_img)
print(f"✅ Grad-CAM:    shape={cam_test.shape}, range=[{cam_test.min():.3f}, {cam_test.max():.3f}]")
print(f"✅ Grad-CAM++:  shape={cam_pp_test.shape}, range=[{cam_pp_test.min():.3f}, {cam_pp_test.max():.3f}]")

## 2. Saliency Maps — gradient bezpośrednio względem pikseli

In [ ]:
def compute_vanilla_saliency(model, img_array):
    """∂prediction / ∂piksele — surowy gradient per-piksel."""
    img_tensor = tf.Variable(img_array[np.newaxis], dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        prediction = model(img_tensor, training=False)
        loss = prediction[0, 0]

    grads = tape.gradient(loss, img_tensor)
    saliency = tf.reduce_max(tf.abs(grads[0]), axis=-1)  # max po RGB
    saliency = saliency / (tf.reduce_max(saliency) + 1e-8)
    return saliency.numpy()


def compute_smoothgrad(model, img_array, n_samples=50, noise_level=0.15):
    """Średnia saliency z N zaszumionych kopii — wygładzona mapa."""
    stdev = noise_level * (img_array.max() - img_array.min())
    accumulated = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)

    for _ in range(n_samples):
        noise = np.random.normal(0, stdev, img_array.shape).astype(np.float32)
        noisy = img_array + noise

        img_tensor = tf.Variable(noisy[np.newaxis], dtype=tf.float32)
        with tf.GradientTape() as tape:
            tape.watch(img_tensor)
            prediction = model(img_tensor, training=False)
            loss = prediction[0, 0]

        grads = tape.gradient(loss, img_tensor)
        accumulated += tf.reduce_max(tf.abs(grads[0]), axis=-1).numpy()

    accumulated /= n_samples
    accumulated /= (accumulated.max() + 1e-8)
    return accumulated


# Test
sal_test = compute_vanilla_saliency(model, test_img)
smooth_test = compute_smoothgrad(model, test_img, n_samples=5)
print(f"✅ Vanilla Saliency: {sal_test.shape}")
print(f"✅ SmoothGrad:       {smooth_test.shape}")

In [ ]:
def overlay_heatmap(img_array, heatmap, colormap='jet', alpha=0.5):
    """Nakłada kolorową heatmapę na obraz."""
    cmap = cm.get_cmap(colormap)
    colored = cmap(heatmap)[:, :, :3]
    img_norm = img_array / 255.0
    return np.clip((1 - alpha) * img_norm + alpha * colored, 0, 1)

In [ ]:
def full_analysis(model, grad_model, image_pil, title=""):
    """
    Pełna analiza: predykcja + Grad-CAM + Grad-CAM++ + Saliency + SmoothGrad.
    Panel 2×4.
    """
    img = preprocess(image_pil)
    prob = predict(model, img)
    label = "AI-Generated" if prob > 0.5 else "Real"
    confidence = max(prob, 1 - prob) * 100
    color = '#e74c3c' if prob > 0.5 else '#2ecc71'

    print(f"  Grad-CAM...", end=" ")
    cam = compute_gradcam(grad_model, img)
    print("✓  Grad-CAM++...", end=" ")
    cam_pp = compute_gradcam_pp(grad_model, img)
    print("✓  Saliency...", end=" ")
    sal = compute_vanilla_saliency(model, img)
    print("✓  SmoothGrad...", end=" ")
    smooth = compute_smoothgrad(model, img, n_samples=50)
    print("✓")

    # ── Panel 2×4 ──
    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    img_uint8 = img.astype(np.uint8)

    # Rząd 1: Grad-CAM
    axes[0, 0].imshow(img_uint8)
    t = f"{label} ({confidence:.1f}%)\nP(AI) = {prob:.4f}"
    if title:
        t = f"{title}\n{t}"
    axes[0, 0].set_title(t, fontsize=12, fontweight='bold', color=color)

    axes[0, 1].imshow(overlay_heatmap(img, cam, 'jet', 0.5))
    axes[0, 1].set_title("Grad-CAM\n← regiony decyzyjne", fontsize=11)

    axes[0, 2].imshow(overlay_heatmap(img, cam_pp, 'jet', 0.5))
    axes[0, 2].set_title("Grad-CAM++\n← lepsza lokalizacja", fontsize=11)

    axes[0, 3].imshow(cam, cmap='jet')
    axes[0, 3].set_title("Heatmap (surowa)\nbez obrazu", fontsize=11)

    # Rząd 2: Saliency
    axes[1, 0].imshow(img_uint8)
    axes[1, 0].set_title("Oryginał\n(referencja)", fontsize=11)

    axes[1, 1].imshow(sal, cmap='hot')
    axes[1, 1].set_title("Vanilla Saliency\n← surowe gradienty pikseli", fontsize=11)

    axes[1, 2].imshow(smooth, cmap='hot')
    axes[1, 2].set_title("SmoothGrad (N=50)\n← wygładzony", fontsize=11)

    axes[1, 3].imshow(overlay_heatmap(img, smooth, 'hot', 0.6))
    axes[1, 3].set_title("SmoothGrad overlay\n← nałożony na zdjęcie", fontsize=11)

    for ax in axes.flatten():
        ax.set_xticks([]); ax.set_yticks([])

    plt.suptitle(
        f"Analiza: {label} | P(AI)={prob:.4f}",
        fontsize=16, fontweight='bold', color=color
    )
    plt.tight_layout()
    plt.show()

    return {
        'prob_ai': float(prob), 'label': label, 'confidence': confidence,
        'heatmaps': {'gradcam': cam, 'gradcam_pp': cam_pp,
                     'saliency': sal, 'smoothgrad': smooth}
    }

print("full_analysis() loaded ✅")

## Testuj na swoich zdjęciach

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

uploader = widgets.FileUpload(accept='image/*', multiple=True, description='Wybierz zdjęcia')
display(HTML("<h3>Kliknij przycisk aby wybrać zdjęcia:</h3>"))
display(uploader)

In [ ]:
# ── Uruchom PO wybraniu zdjęć ──

def extract_upload(fi):
    if isinstance(fi, dict): return fi['content'], fi['name']
    else: return fi.content, fi.name

if not uploader.value:
    print("⚠️ Najpierw wybierz zdjęcia powyżej!")
else:
    results = []
    for file_info in uploader.value:
        content, fname = extract_upload(file_info)
        img_pil = Image.open(io.BytesIO(content))

        print(f"\n{'='*70}")
        print(f"  {fname}")
        print(f"{'='*70}")

        r = full_analysis(model, grad_model, img_pil, title=fname)
        results.append({**r, 'file': fname})

    # Podsumowanie
    print(f"\n\n{'='*70}")
    print(f"{'PODSUMOWANIE':^70}")
    print(f"{'='*70}")
    print(f"{'Plik':<35s} {'Wynik':<16s} {'P(AI)':<10s} {'Pewność'}")
    print(f"{'-'*70}")
    for r in results:
        icon = "🤖" if r['label'] == 'AI-Generated' else "📷"
        print(f"{r['file']:<35s} {icon} {r['label']:<13s} {r['prob_ai']:<10.4f} {r['confidence']:.1f}%")
    print(f"{'='*70}")

## Alternatywa — ścieżka do pliku

In [ ]:
IMAGE_PATH = "kot.png"  # ← zmień na swoją ścieżkę

if os.path.exists(IMAGE_PATH):
    img_pil = Image.open(IMAGE_PATH)
    result = full_analysis(model, grad_model, img_pil, title=IMAGE_PATH)
else:
    print(f"Plik nie istnieje: {IMAGE_PATH}")

## Porównanie wielu zdjęć — widok skompresowany

In [ ]:
def quick_comparison(model, grad_model, image_list, titles=None):
    """1 wiersz = 1 zdjęcie: oryginał + Grad-CAM + SmoothGrad + różnica."""
    n = len(image_list)
    fig, axes = plt.subplots(n, 4, figsize=(20, 4.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i, img_pil in enumerate(image_list):
        img = preprocess(img_pil)
        prob = predict(model, img)
        label = "AI-Generated" if prob > 0.5 else "Real"
        conf = max(prob, 1 - prob) * 100
        color = '#e74c3c' if prob > 0.5 else '#2ecc71'

        cam = compute_gradcam(grad_model, img)
        sal = compute_smoothgrad(model, img, n_samples=30)

        title = titles[i] if titles else f"Image {i+1}"

        axes[i, 0].imshow(img.astype(np.uint8))
        axes[i, 0].set_title(f"{title}\n{label} ({conf:.1f}%)",
                            fontsize=10, fontweight='bold', color=color)

        axes[i, 1].imshow(overlay_heatmap(img, cam, 'jet', 0.5))
        axes[i, 1].set_title("Grad-CAM", fontsize=10)

        axes[i, 2].imshow(overlay_heatmap(img, sal, 'hot', 0.5))
        axes[i, 2].set_title("SmoothGrad", fontsize=10)

        diff = np.abs(cam - sal)
        diff = diff / (diff.max() + 1e-8)
        axes[i, 3].imshow(diff, cmap='coolwarm')
        axes[i, 3].set_title("Różnica CAM vs Saliency", fontsize=10)

    for ax in axes.flatten():
        ax.set_xticks([]); ax.set_yticks([])
    plt.suptitle("Porównanie", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Przykład:
# images = [Image.open(f) for f in ["real.jpg", "ai.jpg"]]
# quick_comparison(model, grad_model, images, ["Real", "AI"])
print("quick_comparison() loaded ✅")

## Jak czytać wyniki?

### Grad-CAM (rząd 1)
Niska rozdzielczość (7×7 upscaled), pokazuje **regiony**.
- Ciepłe obszary = "tu model widzi coś podejrzanego"
- Twarze, ręce, przejścia tło-obiekt → typowe dla artefaktów AI

### Saliency / SmoothGrad (rząd 2)
Wysoka rozdzielczość (per-piksel), pokazuje **detale**.
- SmoothGrad jest czytelniejszy niż vanilla saliency (mniej szumu)
- Ciepło na powtarzalnych wzorcach → periodyczne artefakty generatora
- Ciepło na krawędziach → aliasing, nienaturalne wygładzanie

### Sygnały pewności
- Confidence > 90%: wynik wiarygodny
- 70–90%: dość pewny, warto zweryfikować
- < 70%: model niepewny — nie polegaj na wyniku

### Dla Real:
- Heatmapy słabsze i rozproszone
- Brak silnych hot-spotów
